In [1]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,neutral
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


In [2]:
def email_assistant(email_text):
    text = email_text.lower()

    # Urgent condition
    if "urgent" in text or "submit" in text or "deadline" in text:
        return "notify", "urgent"

    # thank you emails
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"

    # Default case
    else:
        return "respond", "neutral"


In [3]:
predictions = [] 
for _, row in df.iterrows(): 
 action, tone = email_assistant(row["body"]) 
predictions.append({ 
"id": row["id"], 
"predicted_intent": action, 
"predicted_tone": tone 
}) 
pred_df = pd.DataFrame(predictions) 
pred_df.head()  

,id,predicted_intent,predicted_tone
0,200,respond,neutral


In [4]:
df[["predicted_intent", "predicted_tone"]] = df["body"].apply(
    lambda x: pd.Series(email_assistant(x))
)

df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone,predicted_intent,predicted_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,neutral,respond,neutral
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral,respond,neutral


In [5]:
df["intent_correct"] = df["predicted_intent"] == df["ideal_intent"]
df["tone_correct"] = df["predicted_tone"] == df["ideal_tone"]

In [6]:
intent_accuracy = df["intent_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

intent_accuracy, tone_accuracy


(np.float64(92.0), np.float64(92.0))

In [7]:
def evaluate(row):
    score = 0

    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1

    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1

    return score


In [8]:
errors = df[~df["intent_correct"] | ~df["tone_correct"]]
len(errors)
errors.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone,predicted_intent,predicted_tone,intent_correct,tone_correct
8,9,security@bank.com,Account Suspended,"Dear user, we detected a login from a new devi...",high,notify_human,notify,urgent,respond,neutral,False,False
27,28,sales@shop.com,Event Invitation,"Dear user, we detected a login from a new devi...",low,notify_human,notify,urgent,respond,neutral,False,False
40,41,news@techblog.com,Survey,"Dear user, we detected a login from a new devi...",low,notify_human,notify,urgent,respond,neutral,False,False
41,42,orders@ecom.com,Payment Overdue,"Dear user, we detected a login from a new devi...",medium,notify_human,notify,urgent,respond,neutral,False,False
46,47,support@cloud.com,Monthly Report,"Dear user, we detected a login from a new devi...",low,notify_human,notify,urgent,respond,neutral,False,False
